In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle


import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load everything back
import pickle
# 1. load Save graph itself ────────────────────────────────────
G             = pickle.load(open('graph.pkl',         'rb'))

# 2. load graph metrics (dictionaries) ───────────────────
betweenness   = pickle.load(open('betweenness.pkl',   'rb'))
in_degree     = pickle.load(open('in_degree.pkl',     'rb'))
out_degree    = pickle.load(open('out_degree.pkl',    'rb'))
edge_weights  = pickle.load(open('edge_weights.pkl',  'rb'))

# load label encoder ───────────────────────────────────
le            = pickle.load(open('label_encoder.pkl', 'rb'))
train_median  = pickle.load(open('train_median.pkl',  'rb'))


# 4. load models
baseline_final_model = pickle.load(open('baseline_final_model.pkl', 'rb'))
graph_final_model = pickle.load(open('graph_final_model.pkl', 'rb'))

osrm_speed = pickle.load(open('osrm_speed.pkl', 'rb'))
osrm_speed_by_route = pickle.load(open('osrm_speed_by_route.pkl', 'rb'))
name_to_code = pickle.load(open('name_to_code.pkl','rb'))

In [ ]:
!pip install folium ipywidgets --quiet
from google.colab import output
output.enable_custom_widget_manager()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 63.5 MB/s eta 0:00:00


In [ ]:
!pip install pgeocode --quiet

## Step 2 — Extract Lat/Lng for Each Facility
Your `source_name` columns likely contain city info. We need coordinates:

In [ ]:
# Build facility coordinates from name_to_code
# source_name format is like "Mumbai_Andheri_Hub (Maharashtra)"
# We'll geocode using a free API — no key needed

import pgeocode
import re

def get_coords_by_pgeocode(center_code, retries=3):
    numbers = re.findall(r'\d+', center_code)

    for num in numbers:
        if len(num) == 6 and num != '000000':
          try:
            nomi = pgeocode.Nominatim('in')
            query = nomi.query_postal_code(num)
            print(f"{center_code} -> PIN:{num}, lat:{query.latitude}, long:{query.longitude}  ✅")
            return query.latitude, query.longitude
          except Exception as e:
            print(f"  {center_code}: error: {e}")

    print(f"  {center_code} → ❌ skipped")
    return None, None

# ── Build coords for ALL unique facility codes ───────────────
all_codes = list(set(name_to_code.values()))
print(f"Total unique facilities: {len(all_codes)}")

facility_coords = {}
failed          = []

for i, code in enumerate(all_codes):

    if code in facility_coords:
        continue

    lat, lng = get_coords_by_pgeocode(code)

    if lat:
        facility_coords[code] = (lat, lng)
    else:
        failed.append(code)


    # Progress every 50
    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{len(all_codes)}] matched so far: {len(facility_coords)}")

   # time.sleep(0.3)
   # time.sleep(1)  # strict 1 req/sec for Nominatim

print(f"\n✅ Got coords : {len(facility_coords)}")
print(f"❌ Failed     : {len(failed)}")
print(f"Failed codes  : {failed[:10]}...")




Total unique facilities: 1643
IND442101AAB -> PIN:442101, lat:20.703005, long:78.51677  ✅
IND764071AAB -> PIN:764071, lat:19.07761176470588, long:82.67201176470587  ✅
IND532421AAA -> PIN:532421, lat:18.36808333333333, long:83.86065833333333  ✅
IND799008AAA -> PIN:799008, lat:23.8903, long:91.4025  ✅
IND401209AAB -> PIN:401209, lat:19.36, long:73.3279  ✅
IND400612AAA -> PIN:400612, lat:19.1941, long:73.0002  ✅
IND345021AAA -> PIN:345021, lat:27.00675, long:71.19330555555557  ✅
IND201007AAB -> PIN:201007, lat:28.7643, long:77.4856  ✅
IND500055AAA -> PIN:500055, lat:17.3724, long:78.437  ✅
IND400705AAA -> PIN:400705, lat:nan, long:nan  ✅
IND508213AAB -> PIN:508213, lat:17.1993, long:79.5185  ✅
IND201007AAA -> PIN:201007, lat:28.7643, long:77.4856  ✅
IND362265AAA -> PIN:362265, lat:21.3729, long:70.4136  ✅
IND431136AAC -> PIN:431136, lat:20.0291, long:75.2816  ✅
IND758035AAA -> PIN:758035, lat:21.649666666666665, long:85.62267777777778  ✅
IND203205AAA -> PIN:203205, lat:28.379908333333333,

In [ ]:
datacoords = pd.DataFrame(facility_coords).T
datacoords

,0,1
IND442101AAB,20.703005,78.516770
IND764071AAB,19.077612,82.672012
IND532421AAA,18.368083,83.860658
IND799008AAA,23.890300,91.402500
IND401209AAB,19.360000,73.327900
...,...,...
IND226016AAA,26.879230,80.871250
IND813213AAA,25.178200,86.844200
IND608001AAA,11.632700,79.446700
IND132001AAA,29.687500,76.884700


#### handeling if there is `null` values

In [ ]:
# ── See which facilities have null coords ───────────────────
null_facilities = datacoords[datacoords.isnull().any(axis=1)]
print(f"Null facilities: {len(null_facilities)}")
print(null_facilities.index.tolist())

Null facilities: 25
['IND400705AAA', 'IND302014AAB', 'IND275503AAA', 'IND492007AAA', 'IND462021AAA', 'IND263002AAA', 'IND302023AAA', 'IND410209AAA', 'IND401104AAB', 'IND509124AAA', 'IND396232AAA', 'IND632402AAA', 'IND205135AAA', 'IND401104AAA', 'IND686028AAB', 'IND421802AAA', 'IND212501AAA', 'IND713364AAB', 'IND811399AAA', 'IND852118AAA', 'IND142401AAA', 'IND122050AAA', 'IND263401AAA', 'IND441603AAA', 'IND302014AAA']


In [ ]:
# ── Strategy: assign to nearest known city ──────────────────
# Major Indian logistics hubs as fallback

city_fallback = {
    'IND000000ACB' : (28.6139, 77.2090),   # Delhi — likely HQ
    'IND000000ACA' : (28.6139, 77.2090),   # Delhi
}

# For remaining nulls — use state-level centroid
# by looking at what other facilities share same prefix

for code in null_facilities.index.tolist():
    if code in city_fallback:
        facility_coords[code] = city_fallback[code]
        print(f"  {code} → manual fallback ✅")
    else:
        # Try neighbouring PIN codes ± 1
        numbers = re.findall(r'\d+', code)
        found   = False

        for num in numbers:
            if len(num) == 6 and num != '000000':
                nomi = pgeocode.Nominatim('in')
                for delta in [1, -1, 2, -2, 5, -5]:
                    neighbour = str(int(num) + delta).zfill(6)
                    result    = nomi.query_postal_code(neighbour)

                    if pd.notna(result.latitude):
                        facility_coords[code] = (
                            float(result.latitude),
                            float(result.longitude)
                        )
                        print(f"  {code} → neighbour PIN {neighbour} ✅")
                        found = True
                        break
                if found:
                    break

        if not found:
            print(f"  {code} → ❌ still missing")

# ── Rebuild datacoords ───────────────────────────────────────
datacoords = pd.DataFrame(facility_coords).T
datacoords.columns = ['latitude', 'longitude']

print(f"\nNulls remaining: {datacoords.isnull().sum().sum()}")

  IND400705AAA → neighbour PIN 400706 ✅
  IND302014AAB → neighbour PIN 302015 ✅
  IND275503AAA → ❌ still missing
  IND492007AAA → neighbour PIN 492008 ✅
  IND462021AAA → neighbour PIN 462022 ✅
  IND263002AAA → neighbour PIN 263001 ✅
  IND302023AAA → neighbour PIN 302022 ✅
  IND410209AAA → neighbour PIN 410210 ✅
  IND401104AAB → neighbour PIN 401105 ✅
  IND509124AAA → neighbour PIN 509125 ✅
  IND396232AAA → neighbour PIN 396230 ✅
  IND632402AAA → neighbour PIN 632403 ✅
  IND205135AAA → ❌ still missing
  IND401104AAA → neighbour PIN 401105 ✅
  IND686028AAB → ❌ still missing
  IND421802AAA → ❌ still missing
  IND212501AAA → neighbour PIN 212502 ✅
  IND713364AAB → neighbour PIN 713365 ✅
  IND811399AAA → ❌ still missing
  IND852118AAA → neighbour PIN 852116 ✅
  IND142401AAA → ❌ still missing
  IND122050AAA → neighbour PIN 122051 ✅
  IND263401AAA → ❌ still missing
  IND441603AAA → neighbour PIN 441601 ✅
  IND302014AAA → neighbour PIN 302015 ✅

Nulls remaining: 14


#### If Nulls Still Remain After That

In [ ]:
pd.DataFrame(facility_coords).T.isnull().sum()

,0
0,7
1,7


In [ ]:
# ── Remove entries with None/NaN coords ─────────────────────
facility_coords = {
    code: (lat, lng)
    for code, (lat, lng) in facility_coords.items()
    if lat is not None and lng is not None
    and pd.notna(lat) and pd.notna(lng)
}

print(f"Final facilities with valid coords: {len(facility_coords)}")

# ── Rebuild datacoords from clean dict ──────────────────────
datacoords = pd.DataFrame(facility_coords).T
datacoords.columns = ['latitude', 'longitude']

print(f"Nulls remaining: {datacoords.isnull().sum().sum()}")

# ----------------

pickle.dump(facility_coords, open('facility_coords.pkl', 'wb'))
print("Saved ✅")

Final facilities with valid coords: 1610
Nulls remaining: 0
Saved ✅


## start from loading

In [ ]:
facility_coords = pickle.load(open('facility_coords.pkl', 'rb'))
datacoords = pd.DataFrame(facility_coords).T
datacoords.columns = ['latitude', 'longitude']
datacoords.head()

,latitude,longitude
IND442101AAB,20.703005,78.516770
IND764071AAB,19.077612,82.672012
IND532421AAA,18.368083,83.860658
IND799008AAA,23.890300,91.402500
IND401209AAB,19.360000,73.327900


In [ ]:
# ── Helper to find best match ───────────────────────────────
from difflib import get_close_matches

def resolve_facility(user_input):
    cleaned = str(user_input).lower().strip()

    # Exact match
    if cleaned in name_to_code:
        return name_to_code[cleaned], user_input

    # Check if user input is contained in any key
    # e.g. 'mumbai' found in 'mumbai hub - andheri'
    contains_matches = [
        k for k in name_to_code.keys()
        if isinstance(k, str) and cleaned in k
    ]

    if contains_matches:
        print(f"\n  '{user_input}' matched these facilities:")
        for i, match in enumerate(contains_matches[:5]):
            print(f"    [{i+1}] {match.title()}")

        choice = input("  Enter number (or 0 to cancel): ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(contains_matches[:5]):
            chosen = contains_matches[int(choice)-1]
            return name_to_code[chosen], chosen.title()
        else:
            return None, None

    # Fuzzy match as fallback (lowered cutoff)
    valid_keys = [k for k in name_to_code.keys() if isinstance(k, str)]
    close = get_close_matches(
        cleaned,
        valid_keys,
        n        = 5,      # show more options
        cutoff   = 0.25    # very relaxed ← 25% matches required
    )

    if close:
        print(f"\n  '{user_input}' not found exactly.")
        print(f"  Did you mean one of these?")
        for i, match in enumerate(close):
            print(f"    [{i+1}] {match.title()}")

        choice = input("  Enter number (or 0 to cancel): ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(close):
            chosen = close[int(choice)-1]
            return name_to_code[chosen], chosen.title()
        else:
            return None, None

    print(f"  ❌ '{user_input}' not recognized in the network.")
    print(f"  Try checking available cities with: show_available_cities('{user_input}')")
    return None, None

#### Step 3 — distance compute

In [ ]:
import requests

def get_road_distance(lat1, lon1, lat2, lon2):
    """
    Uses OSRM public API — actual road distance
    same engine Delhivery uses internally
    """
    try:
        url = (
            f"http://router.project-osrm.org/route/v1/driving/"
            f"{lon1},{lat1};{lon2},{lat2}"
            f"?overview=false"
        )
        r    = requests.get(url, timeout=10)
        data = r.json()

        if data['code'] == 'Ok':
            route        = data['routes'][0]
            distance_km  = route['distance'] / 1000      # meters → km
            duration_min = route['duration'] / 60        # seconds → mins

            return distance_km, duration_min

    except Exception as e:
        print(f"OSRM error: {e}")

    return None, None

# ── Test ─────────────────────────────────────────────────────
# dist, dur = get_road_distance(28.54978, 77.26512, 12.9846,80.1747)
# print(f"Road distance : {dist:.1f} km")
# print(f"OSRM duration : {dur:.1f} mins")

Road distance : 2108.2 km
OSRM duration : 1591.9 mins


#### haversine_distance for backup

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate road distance approximation in km"""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    straight_line_km = R * c
    # Road distance is typically 1.3x straight line in India
    return straight_line_km * 1.3

In [ ]:
# ── Build code → name lookup ────────────────────────────────
code_to_name = {v: k.title() for k, v in name_to_code.items()}

pickle.dump(code_to_name, open('code_to_name.pkl', 'wb'))
print("Saved ✅")

Saved ✅


#### Step 4 — Full Interactive Map Widget

In [ ]:
import folium
import ipywidgets as widgets
from IPython.display import display, clear_output
from math import radians, sin, cos, sqrt, atan2
from datetime import datetime
import pandas as pd

code_to_name = pickle.load(open('code_to_name.pkl', 'rb'))

# ── All facility names for autocomplete ─────────────────────
all_facility_names = sorted([
    k.title() for k in name_to_code.keys()
    if isinstance(k, str)
])



def build_interactive_map():

    degree_df = pd.DataFrame({
        'facility'   : list(in_degree.keys()),
        'in_degree'  : list(in_degree.values()),
        'out_degree' : [out_degree[n] for n in in_degree.keys()]
    })
    # Convert to dataframe
    betweenness_df = pd.DataFrame({
        'facility'    : list(betweenness.keys()),
        'betweenness' : list(betweenness.values())
    }).sort_values('betweenness', ascending=False)

    # ── Widgets ─────────────────────────────────────────────
    src_input    = widgets.Text(
                     placeholder = 'Type source city (4+ chars)...',
                     description = 'From:',
                     layout      = widgets.Layout(width='320px')
                   )
    dst_input    = widgets.Text(
                     placeholder = 'Type destination city (4+ chars)...',
                     description = 'To:',
                     layout      = widgets.Layout(width='320px')
                   )
    src_dropdown = widgets.Select(
                     options = [],
                     rows    = 4,
                     layout  = widgets.Layout(width='320px', display='none')
                   )
    dst_dropdown = widgets.Select(
                     options = [],
                     rows    = 4,
                     layout  = widgets.Layout(width='320px', display='none')
                   )
    route_select = widgets.Dropdown(
                     options     = ['FTL', 'Carting'],
                     description = 'Route:',
                     layout      = widgets.Layout(width='220px')
                   )
    model_select = widgets.Dropdown(
                     options     = ['Graph-Enhanced Model', 'Baseline Model'],
                     description = 'Model:',
                     layout      = widgets.Layout(width='250px')
                   )
    predict_btn  = widgets.Button(
                     description  = 'Predict ETA',
                     button_style = 'primary',
                     layout       = widgets.Layout(width='150px')
                   )
    output_area  = widgets.Output()
    map_area     = widgets.Output()

    # ── Autocomplete logic ───────────────────────────────────
    def update_suggestions(change, dropdown, input_box):
        typed = change['new'].strip().lower()

        if len(typed) >= 4:
            matches = [name for name in all_facility_names if typed in name.lower()][:8]

            if matches:
                dropdown.options  = matches
                dropdown.layout.display = 'block'
            else:
                dropdown.layout.display = 'none'
        else:
            dropdown.layout.display = 'none'

    def on_src_select(change):
        if change['new']:
            src_input.value         = change['new']
            src_dropdown.layout.display = 'none'

    def on_dst_select(change):
        if change['new']:
            dst_input.value         = change['new']
            dst_dropdown.layout.display = 'none'

    src_input.observe(
        lambda c: update_suggestions(c, src_dropdown, src_input),
        names='value'
    )
    dst_input.observe(
        lambda c: update_suggestions(c, dst_dropdown, dst_input),
        names='value'
    )
    src_dropdown.observe(on_src_select, names='value')
    dst_dropdown.observe(on_dst_select, names='value')

    # ── Draw map ─────────────────────────────────────────────
    def draw_base_map(src_code=None, dst_code=None, result=None):
        m = folium.Map(
            location   = [20.5937, 78.9629],
            zoom_start = 5,
            tiles      = 'OpenStreetMap'
        )


        # Plot all facilities
        hub_risk = betweenness_df.merge(degree_df, on='facility')
        top5 = hub_risk.head(5)['facility'].values

        for code, (lat, lng) in facility_coords.items():
            is_top = code in top5
            place_name = code_to_name.get(code, code)

            folium.CircleMarker(
                location     = [lat, lng],
                radius       = 7 if is_top else 3,
                color        = '#E24B4A' if is_top else '#378ADD',
                fill         = True,
                fill_opacity = 0.8,
                tooltip      = f"{'⚠️ ' if is_top else ''}{place_name}"
            ).add_to(m)

        # Draw route
        if src_code and dst_code:
            src_coords = facility_coords.get(src_code)
            dst_coords = facility_coords.get(dst_code)

            if src_coords and dst_coords:
                folium.Marker(
                    location = src_coords,
                    popup    = code_to_name.get(src_code, src_code),
                    icon     = folium.Icon(color='green', icon='play')
                ).add_to(m)

                folium.Marker(
                    location = dst_coords,
                    popup    = code_to_name.get(dst_code, dst_code),
                    icon     = folium.Icon(color='red', icon='flag')
                ).add_to(m)

                line_color = '#E24B4A' if result and result['delay'] > 1.5 else '#1D9E75'
                folium.PolyLine(
                    locations = [src_coords, dst_coords],
                    color     = line_color,
                    weight    = 4,
                    opacity   = 0.8
                ).add_to(m)

                if result:
                    mid_lat = (src_coords[0] + dst_coords[0]) / 2
                    mid_lng = (src_coords[1] + dst_coords[1]) / 2

                    popup_html = f"""
                    <div style='font-family:sans-serif;padding:10px;min-width:200px'>
                        <b style='font-size:14px'>ETA Prediction</b>
                        <hr style='margin:6px 0'>
                        <b>From:</b> {code_to_name.get(src_code, src_code)}<br>
                        <b>To:</b> {code_to_name.get(dst_code, dst_code)}<br>
                        <b>Route:</b> {result['route']}<br>
                        <b>Model:</b> {result['model']}<br>
                        <b>Distance:</b> {result['distance']:.0f} km<br>
                        <b>OSRM Est:</b> {result['osrm']:.0f} mins<br>
                        <b style='color:#E24B4A'>Predicted:</b>
                        <b style='color:#E24B4A'>{result['eta']:.0f} mins</b><br>
                        <b>Delay:</b> {result['delay']:.2f}x OSRM<br>
                        <b>Est. Travel:</b> ~{result['eta']/60:.1f} hours
                    </div>
                    """
                    folium.Marker(
                        location = [mid_lat, mid_lng],
                        popup    = folium.Popup(popup_html, max_width=240),
                        icon     = folium.DivIcon(
                            html = f"""
                            <div style='background:{line_color};color:white;
                                        padding:4px 8px;border-radius:4px;
                                        font-size:12px;font-weight:bold;
                                        white-space:nowrap'>
                                {result['eta']:.0f} mins
                            </div>"""
                        )
                    ).add_to(m)

        return m

    # ── Predict click ────────────────────────────────────────
    def on_predict(b):
        with output_area:
            clear_output()

            src_text = src_input.value.strip()
            dst_text = dst_input.value.strip()

            if not src_text or not dst_text:
                print("❌ Enter both source and destination")
                return

            src_code, src_name = resolve_facility(src_text)
            dst_code, dst_name = resolve_facility(dst_text)

            if not src_code or not dst_code:
                print("❌ Could not resolve facility names")
                return

            src_coords = facility_coords.get(src_code)
            dst_coords = facility_coords.get(dst_code)

            if not src_coords or not dst_coords:
                print(f"❌ No coordinates for one of the facilities")
                return


            dist_km, osrm_time = get_road_distance(
                src_coords[0], src_coords[1],
                dst_coords[0], dst_coords[1]
            )

            # Fallback if OSRM API fails
            if dist_km is None:
                print("⚠️ OSRM API failed — using haversine fallback")
                dist_km   = haversine_distance(
                                src_coords[0], src_coords[1],
                                dst_coords[0], dst_coords[1]
                            )
                osrm_time = dist_km / osrm_speed_by_route.get(route, osrm_speed)
            # --- end of if----

            route     = route_select.value
            speed     = osrm_speed_by_route.get(route, osrm_speed)


            # Encode route
            #matched_route = match_route_type(route)
            route_encoded = le.transform([route])[0]

             # ──────── Define baseline features (NO graph features) ────────
            baseline_features = [
                'segment_osrm_time',
                'segment_osrm_distance',
                'route_type_encoded',
                'hour_of_day',
                'day_of_week',
                'month'
            ]

            # Graph-enhanced features (adds graph info on top)
            graph_features = baseline_features + [
                'src_betweenness',
                'dst_betweenness',
                'src_in_degree',
                'src_out_degree',
                'dst_in_degree',
                'dst_out_degree',
                'corridor_median_delay'
            ]
            target = 'segment_actual_time'

            now = datetime.now()

            row = {
                'segment_osrm_time'     : osrm_time,
                'segment_osrm_distance' : dist_km,
                'route_type_encoded'    : route_encoded,
                'hour_of_day'           : now.hour,
                'day_of_week'           : now.weekday(),
                'month'                 : now.month,
                'src_betweenness'       : betweenness.get(src_code, 0),
                'dst_betweenness'       : betweenness.get(dst_code, 0),
                'src_in_degree'         : in_degree.get(src_code, 0),
                'src_out_degree'        : out_degree.get(src_code, 0),
                'dst_in_degree'         : in_degree.get(dst_code, 0),
                'dst_out_degree'        : out_degree.get(dst_code, 0),
                'corridor_median_delay' : edge_weights.get(
                                            (src_code, dst_code), train_median
                                          )
            }

            # ── Model selection ──────────────────────────────
            selected = model_select.value
            if selected == 'Baseline Model':
                model_used    = baseline_final_model
                features_used = baseline_features
            else:
                model_used    = graph_final_model
                features_used = graph_features

            pred  = model_used.predict(
                        pd.DataFrame([row])[features_used]
                    )[0]

            result = {
                'eta'      : pred,
                'osrm'     : osrm_time,
                'delay'    : pred / osrm_time,
                'distance' : dist_km,
                'route'    : route,
                'model'    : selected
            }

            print(f"\n{'='*42}")
            print(f"  ✅ ETA PREDICTION")
            print(f"{'='*42}")
            print(f"  Model    : {selected}")
            print(f"  From     : {src_name.title()}")
            print(f"  To       : {dst_name.title()}")
            print(f"  Distance : {dist_km:.0f} km")
            print(f"  OSRM Est : {osrm_time:.0f} mins")
            print(f"  Predicted: {pred:.0f} mins ({pred/60:.1f} hrs)")
            print(f"  Delay    : {result['delay']:.2f}x OSRM")
            print(f"{'='*42}")

            with map_area:
                clear_output()
                display(draw_base_map(src_code, dst_code, result))

    predict_btn.on_click(on_predict)

    # ── Layout ───────────────────────────────────────────────
    controls = widgets.VBox([
        widgets.HBox([
            widgets.VBox([src_input, src_dropdown]),
            widgets.VBox([dst_input, dst_dropdown]),
        ]),
        widgets.HBox([route_select, model_select, predict_btn])
    ])

    with map_area:
        display(draw_base_map())

    display(widgets.VBox([
        widgets.HTML("<h3 style='margin:8px 0'>🗺️ Delhivery ETA Predictor</h3>"),
        controls,
        output_area,
        map_area
    ]))

build_interactive_map()

In [ ]:
build_interactive_map()